<a href="https://colab.research.google.com/github/DarkfinShark/compling-hw/blob/main/fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [1]:
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)
from datasets import load_dataset
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# 1. Загрузка датасета
dataset = load_dataset('ag_news')
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [4]:
# 2. Загрузка модели и токенизатора
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# 3. Токенизация
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets['train']
eval_dataset = tokenized_datasets['test']

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [6]:
# 4. Настройка обучения
training_args = TrainingArguments(
    output_dir='./ag_news_model',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit = 2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

In [7]:
# 5. Обучение
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.200394,0.181882,0.942500
2,0.131197,0.189244,0.946447
3,0.074375,0.237602,0.946579


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=22500, training_loss=0.1501834771050347, metrics={'train_runtime': 6484.6085, 'train_samples_per_second': 55.516, 'train_steps_per_second': 3.47, 'total_flos': 1.6999491211228032e+16, 'train_loss': 0.1501834771050347, 'epoch': 3.0})

In [8]:
# 6. Accuracy на тестовой выборке
eval_results = trainer.evaluate()
print(f'Test accuracy: {eval_results['eval_accuracy']}')

Test accuracy: 0.9465789473684211


In [9]:
# 7. Сохранение модели
trainer.save_model('./ag_news_model')
tokenizer.save_pretrained('./ag_news_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ag_news_model/tokenizer_config.json', './ag_news_model/tokenizer.json')

In [10]:
# 8. Тестирование с использованием pipeline
classifier = pipeline(
    'text-classification',
    model = model,
    tokenizer = tokenizer
)

test_texts = [
    "F1 Australian Grand Prix briefing: George Russell wins as Ferrari’s strategy backfires",
    "Pentagon Officially Notifies Anthropic It Is a ‘Supply Chain Risk’",
    "Australian Leaders Promise to Tighten Gun Laws After Bondi Beach Attack"
]

for text in test_texts:
    result = classifier(text)[0]
    print(f'Text: {text}\nClass: {result['label']}, Confidence: {result['score']}')

Text: F1 Australian Grand Prix briefing: George Russell wins as Ferrari’s strategy backfires
Class: LABEL_1, Confidence: 0.9860097169876099
Text: Pentagon Officially Notifies Anthropic It Is a ‘Supply Chain Risk’
Class: LABEL_2, Confidence: 0.9990143775939941
Text: Australian Leaders Promise to Tighten Gun Laws After Bondi Beach Attack
Class: LABEL_0, Confidence: 0.9995630383491516
